# QQQ Drawdown Detection - Data Download and Feature Engineering
------------------------------------------

### Import the modules

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from scipy.stats import pearsonr
print('Modules are imported.')

### Download QQQ data

In [ ]:
print("Descargando datos de QQQ...")
ticker = "QQQ"
data = yf.download(ticker, start="2014-01-01", end="2024-12-31", progress=False)
print(f"QQQ: {len(data)} registros")

### Download SPY data (for correlation analysis)

In [ ]:
print("Descargando datos de SPY...")
spy_data = yf.download("SPY", start="2014-01-01", end="2024-12-31", progress=False)
print(f"SPY: {len(spy_data)} registros")

### Download VIX data (Volatility Index)

In [ ]:
print("Descargando datos de VIX...")
vix_data = yf.download("^VIX", start="2014-01-01", end="2024-12-31", progress=False)
print(f"VIX: {len(vix_data)} registros")

### Prepare DataFrames (flatten MultiIndex columns)

In [ ]:
# Si data tiene MultiIndex en columnas, aplanarlo
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.droplevel(1)
if isinstance(spy_data.columns, pd.MultiIndex):
    spy_data.columns = spy_data.columns.droplevel(1)
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = vix_data.columns.droplevel(1)

print("DataFrames preparados correctamente")

### Create main DataFrame with basic QQQ columns

In [ ]:
df = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
print(f"DataFrame principal creado con {len(df)} registros")
df.head()

### Feature 1: Add VIX (Volatility Index)

In [ ]:
df['VIX'] = vix_data['Close']
print("✓ Feature VIX agregado")

### Feature 2: RSI (Relative Strength Index) 14 days

In [ ]:
df['RSI_14d'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()
print("✓ Feature RSI_14d agregado")

### Feature 3: MACD (Moving Average Convergence Divergence)

In [ ]:
macd = ta.trend.MACD(close=df['Close'])
df['MACD'] = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Diff'] = macd.macd_diff()
print("✓ Features MACD, MACD_Signal, MACD_Diff agregados")

### Feature 4: Historical Volatility 30 days (annualized)

In [ ]:
df['Returns'] = df['Close'].pct_change()
df['Historical_Vol_30d'] = df['Returns'].rolling(window=30).std() * np.sqrt(252)
print("✓ Feature Historical_Vol_30d agregado")

### Feature 5: Bollinger Bands Width

In [ ]:
bollinger = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['BB_High'] = bollinger.bollinger_hband()
df['BB_Low'] = bollinger.bollinger_lband()
df['BB_Mid'] = bollinger.bollinger_mavg()
df['Bollinger_Band_Width'] = (df['BB_High'] - df['BB_Low']) / df['BB_Mid']
print("✓ Feature Bollinger_Band_Width agregado")

### Feature 6: ATR (Average True Range) 14 days

In [ ]:
df['ATR_14d'] = ta.volatility.AverageTrueRange(
    high=df['High'], 
    low=df['Low'], 
    close=df['Close'], 
    window=14
).average_true_range()
print("✓ Feature ATR_14d agregado")

### Feature 7: Distance to MA200 (percentage)

In [ ]:
df['MA200'] = df['Close'].rolling(window=200).mean()
df['Distance_to_MA200'] = ((df['Close'] - df['MA200']) / df['MA200']) * 100
print("✓ Feature Distance_to_MA200 agregado")

### Feature 8: Volume Ratio (current vs 20-day average)

In [ ]:
df['Volume_MA20'] = df['Volume'].rolling(window=20).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_MA20']
print("✓ Feature Volume_Ratio agregado")

### Feature 9: SPY-QQQ Correlation (rolling 60 days)

In [ ]:
spy_returns = spy_data['Close'].pct_change()
df['SPY_Returns'] = spy_returns

def rolling_correlation(df, window=60):
    corr = []
    for i in range(len(df)):
        if i < window - 1:
            corr.append(np.nan)
        else:
            qqq_ret = df['Returns'].iloc[i-window+1:i+1]
            spy_ret = df['SPY_Returns'].iloc[i-window+1:i+1]
            mask = ~(qqq_ret.isna() | spy_ret.isna())
            if mask.sum() > 10:
                corr.append(qqq_ret[mask].corr(spy_ret[mask]))
            else:
                corr.append(np.nan)
    return corr

df['SPY_QQQ_Correlation'] = rolling_correlation(df, window=60)
print("✓ Feature SPY_QQQ_Correlation agregado")

### Feature 10: 20-day Return

In [ ]:
df['Return_20d'] = df['Close'].pct_change(periods=20) * 100
print("✓ Feature Return_20d agregado")

### Select final features for analysis

In [ ]:
features_df = df[[
    'Open', 'High', 'Low', 'Close', 'Volume',
    'VIX',
    'RSI_14d',
    'MACD', 'MACD_Signal', 'MACD_Diff',
    'Historical_Vol_30d',
    'Bollinger_Band_Width',
    'ATR_14d',
    'Distance_to_MA200',
    'Volume_Ratio',
    'SPY_QQQ_Correlation',
    'Return_20d'
]].copy()

print(f"\n✅ Proceso de feature engineering completado!")
print(f"Total de features: {len(features_df.columns)}")

### Save complete dataset (with NaN values)

In [ ]:
features_df.to_csv("QQQ_features_2014_2024.csv", index=True)
print("Archivo guardado: QQQ_features_2014_2024.csv")

### Save clean dataset (without NaN values)

In [ ]:
features_clean = features_df.dropna()
features_clean.to_csv("QQQ_features_clean_2014_2024.csv", index=True)
print(f"Archivo guardado: QQQ_features_clean_2014_2024.csv")
print(f"Registros limpios (sin NaN): {len(features_clean)}")

### Display summary statistics

In [ ]:
print("="*60)
print("RESUMEN DE FEATURES")
print("="*60)
features_clean.describe()

### Display first 5 rows

In [ ]:
print("="*60)
print("PRIMERAS 5 FILAS")
print("="*60)
features_clean.head()

### Check missing values by column

In [ ]:
print("="*60)
print("VALORES FALTANTES POR COLUMNA")
print("="*60)
features_df.isnull().sum()